In [ ]:
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time
import os
from datetime import datetime, timedelta, timezone

BASE_URL = "https://api.hitbtc.com/api/3/public/trades/{symbol}"

COINS = ["ETH", "BTC", "BCH", "BSV", "LTC", "XMR", "ZEC", "DASH"]
QUOTE = "USDT"  # change to "USD" if a coin does not have a USDT pair on HitBTC

START_DATE = datetime(2019, 11, 1, tzinfo=timezone.utc)
END_DATE = datetime(2026, 1, 1, tzinfo=timezone.utc)

OUTPUT_DIR = "data/HitBTC_Intraday_Data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LIMIT = 1000          # HitBTC max trades per request
REQUEST_SLEEP = 0.25      # seconds between requests to respect rate limits
CHUNK_DAYS = 1            # internal fetch window; does not affect output file structure

# Fixed schema so every row group written to the Parquet file has consistent types
SCHEMA = pa.schema([
    ("id", pa.int64()),
    ("price", pa.float64()),
    ("qty", pa.float64()),
    ("side", pa.string()),
    ("timestamp", pa.timestamp("ms", tz="UTC")),
    ("coin", pa.string()),
])


def fetch_trades_for_window(symbol, from_ts_ms, till_ts_ms, session):
    """
    Fetch all trades for a symbol between from_ts_ms and till_ts_ms (inclusive),
    paginating forward using the timestamp of the last trade received.
    """
    all_trades = []
    current_from = from_ts_ms

    while True:
        params = {
            "sort": "ASC",
            "by": "timestamp",
            "from": current_from,
            "till": till_ts_ms,
            "limit": MAX_LIMIT,
        }
        resp = session.get(BASE_URL.format(symbol=symbol), params=params, timeout=30)

        if resp.status_code == 429:
            print(f"Rate limited on {symbol}, sleeping 5s...")
            time.sleep(5)
            continue

        if resp.status_code != 200:
            print(f"Error {resp.status_code} for {symbol} at {current_from}: {resp.text[:200]}")
            break

        batch = resp.json()
        if not batch:
            break

        all_trades.extend(batch)

        last_ts = batch[-1]["timestamp"]
        last_ts_ms = int(pd.Timestamp(last_ts).timestamp() * 1000)

        if len(batch) < MAX_LIMIT or last_ts_ms >= till_ts_ms:
            break

        current_from = last_ts_ms + 1
        time.sleep(REQUEST_SLEEP)

    return all_trades


def daterange_chunks(start, end, days):
    current = start
    while current < end:
        chunk_end = min(current + timedelta(days=days), end)
        yield current, chunk_end
        current = chunk_end


def trades_to_table(trades, coin):
    """
    Converts a list of raw trade dicts into a PyArrow Table matching the
    fixed SCHEMA, ready to be appended as a new row group in the Parquet file.
    """
    df = pd.DataFrame(trades)
    df["price"] = df["price"].astype(float)
    df["qty"] = df["qty"].astype(float)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["coin"] = coin.lower()
    df = df[["id", "price", "qty", "side", "timestamp", "coin"]]
    return pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False)


def download_coin_trades(coin, quote, start_date, end_date):
    """
    Downloads all trades for one coin across the full date range and writes
    them incrementally as row groups into a single Parquet file for that coin
    (avoids holding years of tick data in memory, and produces a much smaller
    file on disk than the equivalent CSV due to Parquet's columnar compression).
    """
    symbol = f"{coin}{quote}"
    session = requests.Session()

    output_file = os.path.join(OUTPUT_DIR, f"Intra_{coin.lower()}_full.parquet")
    if os.path.exists(output_file):
        os.remove(output_file)  # start clean each run

    writer = None
    total_trades = 0

    try:
        for window_start, window_end in daterange_chunks(start_date, end_date, CHUNK_DAYS):
            from_ms = int(window_start.timestamp() * 1000)
            till_ms = int(window_end.timestamp() * 1000)

            trades = fetch_trades_for_window(symbol, from_ms, till_ms, session)

            if trades:
                table = trades_to_table(trades, coin)

                if writer is None:
                    writer = pq.ParquetWriter(output_file, SCHEMA, compression="snappy")

                writer.write_table(table)
                total_trades += len(trades)
                print(f"{coin}: wrote {len(trades)} trades for {window_start.date()} (total so far: {total_trades})")
            else:
                print(f"{coin}: no trades for {window_start.date()}")

            time.sleep(REQUEST_SLEEP)
    finally:
        if writer is not None:
            writer.close()

    print(f"=== {coin}: finished, {total_trades} total trades saved to {output_file} ===")


if __name__ == "__main__":
    for coin in COINS:
        print(f"=== Downloading {coin}{QUOTE} trades from {START_DATE.date()} to {END_DATE.date()} ===")
        download_coin_trades(coin, QUOTE, START_DATE, END_DATE)